# Notebook 10 — Tree-Based Baselines (RF + XGBoost)

Four experiments: Random Forest and XGBoost, each on the full feature set and
the price-only subset. These are the non-sequential baselines that the
LSTM+Attention model (Notebook 11) must beat to justify its complexity.

**Pipeline position.**

`features_targets.parquet` (Step 4) → **this notebook** → `eval_results_baselines.json` + feature importance plots.


## Setup

In [1]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

try:
    from xgboost import XGBClassifier
except ImportError:
    !pip install xgboost -q
    from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
print("Setup complete.")

Setup complete.


In [2]:
# --- CONFIG ---
INPUT_DIR  = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"
FIG_DIR    = f"{OUTPUT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

DATA_PATH    = f"{INPUT_DIR}/datasets/leev75/pfe-step41/features_targets.parquet"
RESULTS_PATH = f"{OUTPUT_DIR}/eval_results_baselines.json"

TARGET_COL   = "target_X1.0"
LABEL_MAP    = {"BUY": 0, "HOLD": 1, "SELL": 2}
LABEL_NAMES  = ["BUY", "HOLD", "SELL"]

META_COLS = ["trading_day", "split", "ret_continuous",
             "target_X0.5", "target_X1.0", "target_X1.5", "target_X2.0"]

# Price-only columns: OHLC + volume_shares.
PRICE_BASE = ["open", "high", "low", "close", "adj_close", "volume_shares"]

SEED = 42
N_BOOTSTRAP = 1000

np.random.seed(SEED)
print("Config loaded.")

Config loaded.


## 1. Load data and define feature sets

In [3]:
df = pd.read_parquet(DATA_PATH)
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")

train = df[df["split"] == "train"].copy()
test  = df[df["split"] == "test"].copy()
print(f"Train: {len(train)}, Test: {len(test)}")

# Full feature set: everything except meta columns.
all_features = [c for c in df.columns if c not in META_COLS]
print(f"Full feature set: {len(all_features)} columns")

# Price-only: columns whose name starts with a price base column or is a price base column.
price_features = [c for c in all_features
                  if any(c == p or c.startswith(p + "_") for p in PRICE_BASE)]
print(f"Price-only feature set: {len(price_features)} columns")

# Encode target.
for split_df in [train, test]:
    split_df["y"] = split_df[TARGET_COL].map(LABEL_MAP)

y_train = train["y"].values
y_test  = test["y"].values

# Class distribution.
print()
print("Train class distribution:")
print(train[TARGET_COL].value_counts())
print()
print("Test class distribution:")
print(test[TARGET_COL].value_counts())

Loaded: 995 rows × 237 columns
Train: 746, Test: 249
Full feature set: 230 columns
Price-only feature set: 5 columns

Train class distribution:
target_X1.0
BUY     308
SELL    306
HOLD    132
Name: count, dtype: int64

Test class distribution:
target_X1.0
BUY     110
SELL     87
HOLD     52
Name: count, dtype: int64


In [4]:
# Scale features (required for LSTM in Nb11; applied here too for consistency).
scaler = StandardScaler()

X_train_full  = scaler.fit_transform(train[all_features])
X_test_full   = scaler.transform(test[all_features])

# Price-only: separate scaler to avoid info leakage from sentiment columns.
scaler_price = StandardScaler()
X_train_price = scaler_price.fit_transform(train[price_features])
X_test_price  = scaler_price.transform(test[price_features])

print(f"X_train_full:  {X_train_full.shape}")
print(f"X_train_price: {X_train_price.shape}")

X_train_full:  (746, 230)
X_train_price: (746, 5)


## 2. Evaluation helpers

In [5]:
def compute_sharpe(y_true_labels, y_pred_labels, ret_continuous, annualise=252):
    """Sharpe Ratio from predicted BUY/SELL/HOLD labels.

    Strategy: BUY -> +1, SELL -> -1, HOLD -> 0.
    Daily return = position * actual return.
    """
    position_map = {"BUY": 1.0, "SELL": -1.0, "HOLD": 0.0}
    positions = np.array([position_map[l] for l in y_pred_labels])
    daily_ret = positions * ret_continuous
    if daily_ret.std() == 0:
        return 0.0
    return float(daily_ret.mean() / daily_ret.std() * np.sqrt(annualise))


def evaluate_experiment(y_true_int, y_pred_int, y_true_labels, y_pred_labels,
                         ret_continuous):
    """Compute all metrics for one experiment."""
    da     = accuracy_score(y_true_int, y_pred_int)
    f1     = f1_score(y_true_int, y_pred_int, average="macro")
    sharpe = compute_sharpe(y_true_labels, y_pred_labels, ret_continuous)

    prec = precision_score(y_true_int, y_pred_int, average=None, labels=[0,1,2], zero_division=0)
    rec  = recall_score(y_true_int, y_pred_int, average=None, labels=[0,1,2], zero_division=0)

    return {
        "DA": da, "macro_F1": f1, "Sharpe": sharpe,
        "precision_BUY": prec[0], "precision_HOLD": prec[1], "precision_SELL": prec[2],
        "recall_BUY": rec[0], "recall_HOLD": rec[1], "recall_SELL": rec[2],
    }


def bootstrap_ci(y_true_int, y_pred_int, y_true_labels, y_pred_labels,
                  ret_continuous, n_boot=N_BOOTSTRAP):
    """Bootstrap 95% CIs for all metrics."""
    n = len(y_true_int)
    boot_results = []
    for _ in range(n_boot):
        idx = np.random.randint(0, n, size=n)
        res = evaluate_experiment(
            y_true_int[idx], y_pred_int[idx],
            [y_true_labels[i] for i in idx],
            [y_pred_labels[i] for i in idx],
            ret_continuous[idx],
        )
        boot_results.append(res)

    boot_df = pd.DataFrame(boot_results)
    summary = {}
    for metric in boot_df.columns:
        vals = boot_df[metric].values
        summary[metric] = {
            "point": float(evaluate_experiment(
                y_true_int, y_pred_int, y_true_labels, y_pred_labels, ret_continuous
            )[metric]),
            "ci_low": float(np.percentile(vals, 2.5)),
            "ci_high": float(np.percentile(vals, 97.5)),
        }
    return summary


def mcnemar_test(y_true, y_pred_a, y_pred_b):
    """McNemar's test comparing two models."""
    from scipy.stats import chi2
    correct_a = (y_pred_a == y_true)
    correct_b = (y_pred_b == y_true)
    # b correct, a wrong
    b_not_a = (~correct_a & correct_b).sum()
    # a correct, b wrong
    a_not_b = (correct_a & ~correct_b).sum()
    if b_not_a + a_not_b == 0:
        return {"statistic": 0.0, "p_value": 1.0}
    stat = (abs(b_not_a - a_not_b) - 1) ** 2 / (b_not_a + a_not_b)
    p = 1 - chi2.cdf(stat, df=1)
    return {"statistic": float(stat), "p_value": float(p)}


print("Evaluation helpers defined.")

Evaluation helpers defined.


## 3. Random Forest

In [6]:
rf_param_grid = {
    "n_estimators": [200, 500],
    "max_depth": [10, 20, None],
    "min_samples_leaf": [5, 10, 20],
}

tscv = TimeSeriesSplit(n_splits=5)

all_results = {}
rf_preds_labels = {}   # capture test-set predictions for dashboard

for feat_label, X_tr, X_te in [("full", X_train_full, X_test_full),
                                 ("price", X_train_price, X_test_price)]:
    print(f"\n--- RF-{feat_label} ---")
    rf = RandomForestClassifier(class_weight="balanced", random_state=SEED, n_jobs=-1)
    gs = GridSearchCV(rf, rf_param_grid, cv=tscv, scoring="f1_macro",
                      n_jobs=-1, verbose=0)
    gs.fit(X_tr, y_train)
    print(f"Best params: {gs.best_params_}")
    print(f"Best CV macro-F1: {gs.best_score_:.4f}")

    y_pred = gs.predict(X_te)
    y_pred_labels = [LABEL_NAMES[i] for i in y_pred]
    y_true_labels = [LABEL_NAMES[i] for i in y_test]

    results = bootstrap_ci(y_test, y_pred, y_true_labels, y_pred_labels,
                            test["ret_continuous"].values)
    results["best_params"] = gs.best_params_
    all_results[f"RF_{feat_label}"] = results
    rf_preds_labels[feat_label] = y_pred_labels   # for dashboard

    print(f"DA:     {results['DA']['point']:.4f} [{results['DA']['ci_low']:.4f}, {results['DA']['ci_high']:.4f}]")
    print(f"F1:     {results['macro_F1']['point']:.4f} [{results['macro_F1']['ci_low']:.4f}, {results['macro_F1']['ci_high']:.4f}]")
    print(f"Sharpe: {results['Sharpe']['point']:.4f} [{results['Sharpe']['ci_low']:.4f}, {results['Sharpe']['ci_high']:.4f}]")

    # Save feature importance for full feature set.
    if feat_label == "full":
        fi = pd.Series(gs.best_estimator_.feature_importances_, index=all_features)
        fi = fi.nlargest(30)
        fig, ax = plt.subplots(figsize=(10, 8))
        fi.plot.barh(ax=ax)
        ax.set_title("Random Forest — Top 30 Feature Importances")
        ax.set_xlabel("Importance")
        plt.tight_layout()
        fig.savefig(f"{FIG_DIR}/feat_importance_rf.png", dpi=150)
        plt.close(fig)
        print(f"Saved feature importance plot.")
        all_results["RF_full"]["feature_importance_top10"] = fi.head(10).to_dict()


--- RF-full ---
Best params: {'max_depth': 10, 'min_samples_leaf': 10, 'n_estimators': 200}
Best CV macro-F1: 0.3047
DA:     0.3454 [0.2851, 0.4016]
F1:     0.1809 [0.1538, 0.2105]
Sharpe: -2.5024 [-4.4940, -0.6566]
Saved feature importance plot.

--- RF-price ---
Best params: {'max_depth': 10, 'min_samples_leaf': 5, 'n_estimators': 200}
Best CV macro-F1: 0.3351
DA:     0.5141 [0.4538, 0.5744]
F1:     0.3925 [0.3472, 0.4365]
Sharpe: 4.6457 [2.8574, 6.5852]


## 4. XGBoost

In [7]:
xgb_param_grid = {
    "n_estimators": [200, 500],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.9],
}

# Compute sample weights for class imbalance.
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight("balanced", y_train)
xgb_preds_labels = {}   # capture test-set predictions for dashboard

for feat_label, X_tr, X_te in [("full", X_train_full, X_test_full),
                                 ("price", X_train_price, X_test_price)]:
    print(f"\n--- XGB-{feat_label} ---")
    xgb = XGBClassifier(
        eval_metric="mlogloss", use_label_encoder=False,
        random_state=SEED, n_jobs=-1, verbosity=0,
    )
    gs = GridSearchCV(xgb, xgb_param_grid, cv=tscv, scoring="f1_macro",
                      n_jobs=-1, verbose=0)
    gs.fit(X_tr, y_train, sample_weight=sample_weights)
    print(f"Best params: {gs.best_params_}")
    print(f"Best CV macro-F1: {gs.best_score_:.4f}")

    y_pred = gs.predict(X_te)
    y_pred_labels = [LABEL_NAMES[i] for i in y_pred]
    y_true_labels = [LABEL_NAMES[i] for i in y_test]

    results = bootstrap_ci(y_test, y_pred, y_true_labels, y_pred_labels,
                            test["ret_continuous"].values)
    results["best_params"] = {k: (int(v) if isinstance(v, (np.integer,)) else v)
                               for k, v in gs.best_params_.items()}
    all_results[f"XGB_{feat_label}"] = results
    xgb_preds_labels[feat_label] = y_pred_labels   # for dashboard

    print(f"DA:     {results['DA']['point']:.4f} [{results['DA']['ci_low']:.4f}, {results['DA']['ci_high']:.4f}]")
    print(f"F1:     {results['macro_F1']['point']:.4f} [{results['macro_F1']['ci_low']:.4f}, {results['macro_F1']['ci_high']:.4f}]")
    print(f"Sharpe: {results['Sharpe']['point']:.4f} [{results['Sharpe']['ci_low']:.4f}, {results['Sharpe']['ci_high']:.4f}]")

    if feat_label == "full":
        fi = pd.Series(gs.best_estimator_.feature_importances_, index=all_features)
        fi = fi.nlargest(30)
        fig, ax = plt.subplots(figsize=(10, 8))
        fi.plot.barh(ax=ax)
        ax.set_title("XGBoost — Top 30 Feature Importances")
        ax.set_xlabel("Importance")
        plt.tight_layout()
        fig.savefig(f"{FIG_DIR}/feat_importance_xgb.png", dpi=150)
        plt.close(fig)
        print(f"Saved feature importance plot.")
        all_results["XGB_full"]["feature_importance_top10"] = fi.head(10).to_dict()


--- XGB-full ---
Best params: {'learning_rate': 0.01, 'max_depth': 8, 'n_estimators': 200, 'subsample': 0.9}
Best CV macro-F1: 0.3215
DA:     0.3896 [0.3333, 0.4499]
F1:     0.2889 [0.2370, 0.3398]
Sharpe: -0.7261 [-2.6411, 1.2074]
Saved feature importance plot.

--- XGB-price ---
Best params: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200, 'subsample': 0.9}
Best CV macro-F1: 0.3764
DA:     0.5301 [0.4699, 0.5904]
F1:     0.3996 [0.3559, 0.4412]
Sharpe: 5.7070 [3.5937, 7.7767]


## 5. Buy-and-Hold baseline

In [8]:
ret = test["ret_continuous"].values
bnh_daily = ret  # always long
bnh_sharpe = float(bnh_daily.mean() / bnh_daily.std() * np.sqrt(252)) if bnh_daily.std() > 0 else 0.0
print(f"Buy-and-Hold Sharpe (2023 test): {bnh_sharpe:.4f}")
all_results["Buy_and_Hold"] = {"Sharpe": {"point": bnh_sharpe}}

Buy-and-Hold Sharpe (2023 test): 2.4679


## 6. Save results

In [9]:
# Serialise numpy types for JSON.
def _convert(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

clean = json.loads(json.dumps(all_results, default=_convert))
with open(RESULTS_PATH, "w") as f:
    json.dump(clean, f, indent=2)

print(f"Saved {RESULTS_PATH}")
print(f"Experiments: {list(all_results.keys())}")

Saved /kaggle/working/eval_results_baselines.json
Experiments: ['RF_full', 'RF_price', 'XGB_full', 'XGB_price', 'Buy_and_Hold']


## 7. Summary table

In [10]:
print(f"{'Experiment':<18s} {'DA':>8s} {'Sharpe':>8s} {'macro-F1':>8s}")
print("-" * 50)
for name in ["RF_full", "RF_price", "XGB_full", "XGB_price"]:
    r = all_results[name]
    print(f"{name:<18s} "
          f"{r['DA']['point']:8.4f} "
          f"{r['Sharpe']['point']:8.4f} "
          f"{r['macro_F1']['point']:8.4f}")
print(f"{'Buy_and_Hold':<18s} {'---':>8s} {all_results['Buy_and_Hold']['Sharpe']['point']:8.4f} {'---':>8s}")

Experiment               DA   Sharpe macro-F1
--------------------------------------------------
RF_full              0.3454  -2.5024   0.1809
RF_price             0.5141   4.6457   0.3925
XGB_full             0.3896  -0.7261   0.2889
XGB_price            0.5301   5.7070   0.3996
Buy_and_Hold            ---   2.4679      ---


---

**Done.** Results saved to `eval_results_baselines.json`. Notebook 11 will load
this file and produce the consolidated 2×4 ablation table.